# Quarantine Data Processing & Recovery

This notebook provides strategies and code for processing quarantined data records.

## Current Quarantine Behavior

```
Bad data → silver_telemetry_quarantine → Isolated (never flows to gold)
```

**Problem**: Quarantined records are lost forever with no recovery mechanism.

## Processing Patterns

### 1. **Manual Review & Reprocessing**
   - Analyze quarantine reasons
   - Fix correctable issues
   - Reinsert into silver_telemetry_enriched

### 2. **Automated Retry Logic**
   - Retry transient failures (e.g., schema evolution)
   - Apply fixes programmatically
   - Move fixed records back to main pipeline

### 3. **Dead Letter Queue (DLQ)**
   - Archive permanently unfixable records
   - Keep quarantine table for recent/fixable issues only
   - Maintain audit trail

### 4. **Data Quality Feedback Loop**
   - Report patterns to upstream systems
   - Fix data quality at source
   - Reduce future quarantine rates

## This Notebook Covers

1. ✅ Analyze quarantine data by reason
2. ✅ Identify fixable vs permanent issues
3. ✅ Fix common data quality problems
4. ✅ Reprocess corrected records
5. ✅ Automated retry workflow
6. ✅ Monitoring and alerting

In [0]:
# Quarantine processing configuration
from pyspark.sql import functions as F
from datetime import datetime, timedelta

CATALOG = "workspace"
SCHEMA = "hive_video_analytics"

# Tables
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_telemetry_quarantine"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_telemetry_enriched"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_telemetry_raw"

# Processing options
LOOKBACK_DAYS = 7  # How far back to look for fixable quarantine records
BATCH_SIZE = 1000  # Records to process per batch

print(f"✅ Configuration loaded")
print(f"Quarantine table: {QUARANTINE_TABLE}")
print(f"Lookback window: {LOOKBACK_DAYS} days")

In [0]:
%sql
-- Understand what types of data quality issues are occurring
-- This helps prioritize which issues to fix first

SELECT 
  quarantine_reason,
  COUNT(*) as issue_count,
  COUNT(DISTINCT customerId) as affected_customers,
  COUNT(DISTINCT eventDate) as affected_dates,
  MIN(quarantine_timestamp) as first_seen,
  MAX(quarantine_timestamp) as last_seen,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_of_total
FROM workspace.hive_video_analytics.silver_telemetry_quarantine
WHERE eventDate >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY quarantine_reason
ORDER BY issue_count DESC;

In [0]:
%sql
-- Categorize quarantine reasons by fixability
-- Some issues can be fixed programmatically, others cannot

WITH quarantine_summary AS (
  SELECT 
    quarantine_reason,
    COUNT(*) as issue_count,
    CASE 
      -- Fixable issues: can be corrected programmatically
      WHEN quarantine_reason LIKE '%negative%' THEN 'Fixable: Apply ABS() or set to 0'
      WHEN quarantine_reason LIKE '%null%' THEN 'Fixable: Fill with defaults'
      WHEN quarantine_reason LIKE '%invalid format%' THEN 'Fixable: Parse and transform'
      WHEN quarantine_reason LIKE '%out of range%' THEN 'Fixable: Cap to valid range'
      
      -- Transient issues: retry with no changes
      WHEN quarantine_reason LIKE '%timeout%' THEN 'Transient: Retry'
      WHEN quarantine_reason LIKE '%schema%' THEN 'Transient: Schema evolved, retry'
      
      -- Permanent issues: cannot be fixed
      WHEN quarantine_reason LIKE '%corrupt%' THEN 'Permanent: Corrupt data'
      WHEN quarantine_reason LIKE '%malformed%' THEN 'Permanent: Malformed JSON'
      
      ELSE 'Unknown: Needs manual review'
    END as fixability,
    
    CASE 
      WHEN quarantine_reason LIKE '%negative%' THEN 'HIGH'
      WHEN quarantine_reason LIKE '%null%' THEN 'HIGH'
      WHEN quarantine_reason LIKE '%timeout%' THEN 'HIGH'
      WHEN quarantine_reason LIKE '%invalid format%' THEN 'MEDIUM'
      WHEN quarantine_reason LIKE '%out of range%' THEN 'MEDIUM'
      ELSE 'LOW'
    END as fix_priority
  FROM workspace.hive_video_analytics.silver_telemetry_quarantine
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
  GROUP BY quarantine_reason
)
SELECT 
  fixability,
  fix_priority,
  COUNT(*) as distinct_reasons,
  SUM(issue_count) as total_records
FROM quarantine_summary
GROUP BY fixability, fix_priority
ORDER BY 
  CASE fix_priority 
    WHEN 'HIGH' THEN 1 
    WHEN 'MEDIUM' THEN 2 
    WHEN 'LOW' THEN 3 
  END,
  total_records DESC;

In [0]:
# Look at actual quarantined records to understand the issues
quarantine_df = spark.table(QUARANTINE_TABLE)

print("📊 Quarantine Table Sample (last 7 days):\n")
recent_quarantine = quarantine_df.filter(
    F.col("eventDate") >= F.current_date() - F.expr("INTERVAL 7 DAYS")
)

record_count = recent_quarantine.count()
print(f"Total quarantine records (last 7 days): {record_count:,}\n")

if record_count > 0:
    # Show sample by quarantine reason
    print("Sample records by quarantine reason:\n")
    display(recent_quarantine.groupBy("quarantine_reason").agg(
        F.count("*").alias("count"),
        F.first("customerId").alias("sample_customerId"),
        F.first("eventDate").alias("sample_eventDate")
    ).orderBy(F.desc("count")))
    
    # Show detailed sample
    print("\nDetailed sample (first 5 records):\n")
    display(recent_quarantine.select(
        "quarantine_reason",
        "quarantine_timestamp",
        "customerId",
        "eventDate",
        "original_data"
    ).limit(5))
else:
    print("✅ No quarantine records found in last 7 days - excellent data quality!")

In [0]:
# Example fix functions for common data quality issues
# Customize these based on your actual quarantine reasons

def fix_negative_values(df):
    """
    Fix negative values in numeric columns by taking absolute value.
    """
    print("🔧 Fixing negative values...")
    
    numeric_cols = ["buffering_count", "buffering_time_ms", "total_source_data", "total_p2p_data"]
    
    fixed_df = df
    for col in numeric_cols:
        if col in df.columns:
            fixed_df = fixed_df.withColumn(col, F.abs(F.col(col)))
    
    return fixed_df

def fix_null_values(df):
    """
    Fill null values with sensible defaults.
    """
    print("🔧 Fixing null values...")
    
    # Fill numeric nulls with 0
    numeric_defaults = {
        "buffering_count": 0,
        "buffering_time_ms": 0,
        "total_source_data": 0,
        "total_p2p_data": 0
    }
    
    # Fill string nulls with 'unknown'
    string_defaults = {
        "quality_level": "unknown"
    }
    
    fixed_df = df.fillna({**numeric_defaults, **string_defaults})
    
    return fixed_df

def fix_out_of_range(df):
    """
    Cap values to valid ranges.
    """
    print("🔧 Fixing out-of-range values...")
    
    fixed_df = df
    
    # Cap buffering time to 1 hour (3600 seconds)
    if "buffering_time_sec" in df.columns:
        fixed_df = fixed_df.withColumn(
            "buffering_time_sec",
            F.when(F.col("buffering_time_sec") > 3600, 3600)
             .otherwise(F.col("buffering_time_sec"))
        )
    
    return fixed_df

def apply_all_fixes(df):
    """
    Apply all fix functions in sequence.
    """
    print("\n🔧 Applying all fixes...\n")
    
    fixed_df = df
    fixed_df = fix_negative_values(fixed_df)
    fixed_df = fix_null_values(fixed_df)
    fixed_df = fix_out_of_range(fixed_df)
    
    print("\n✅ All fixes applied\n")
    return fixed_df

print("✅ Fix functions defined")
print("\nAvailable functions:")
print("  - fix_negative_values(df)")
print("  - fix_null_values(df)")
print("  - fix_out_of_range(df)")
print("  - apply_all_fixes(df)")

In [0]:
# DRY RUN: Process quarantine records without writing to silver
# Review the results before actually reprocessing

print("🔍 DRY RUN: Processing quarantine records\n")
print("=" * 80)

# Load recent quarantine records
quarantine_df = spark.table(QUARANTINE_TABLE).filter(
    F.col("eventDate") >= F.current_date() - F.expr(f"INTERVAL {LOOKBACK_DAYS} DAYS")
)

original_count = quarantine_df.count()
print(f"📊 Quarantine records to process: {original_count:,}\n")

if original_count == 0:
    print("✅ No quarantine records to process!")
else:
    # Reconstruct records in silver schema format
    # NOTE: You'll need to adjust this based on what columns are in your quarantine table
    reconstructed_df = quarantine_df.select(
        "customerId",
        "contentId",
        "clientId",
        "timestamp_server",
        "timestamp_agent",
        "buffering_count",
        "buffering_time_ms",
        "quality_level",
        "eventDate"
        # Add other columns from your quarantine table
    )
    
    # Apply fixes
    fixed_df = apply_all_fixes(reconstructed_df)
    
    # Add derived columns (matching silver schema)
    fixed_df = fixed_df.withColumn(
        "buffering_time_sec",
        F.col("buffering_time_ms") / 1000.0
    )
    
    # Validate fixed records against silver expectations
    print("\n📋 Validating fixed records...\n")
    
    # Check for remaining nulls in required columns
    required_cols = ["customerId", "contentId", "clientId", "quality_level"]
    null_checks = []
    for col in required_cols:
        null_count = fixed_df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            null_checks.append(f"{col}: {null_count} nulls")
            print(f"  ⚠️ {col} has {null_count} null values")
        else:
            print(f"  ✓ {col} has no nulls")
    
    # Check for negative values
    negative_buffering = fixed_df.filter(F.col("buffering_count") < 0).count()
    if negative_buffering > 0:
        print(f"  ⚠️ {negative_buffering} records still have negative buffering_count")
    else:
        print(f"  ✓ No negative buffering_count values")
    
    # Calculate success rate
    validated_count = fixed_df.filter(
        F.col("customerId").isNotNull() &
        F.col("quality_level").isNotNull() &
        (F.col("buffering_count") >= 0)
    ).count()
    
    success_rate = (validated_count / original_count * 100) if original_count > 0 else 0
    
    print(f"\n{'=' * 80}")
    print(f"📊 DRY RUN RESULTS:")
    print(f"{'=' * 80}")
    print(f"Original quarantine records: {original_count:,}")
    print(f"Successfully fixed records:  {validated_count:,}")
    print(f"Still problematic records:   {original_count - validated_count:,}")
    print(f"Success rate:                {success_rate:.1f}%")
    print(f"\n⚠️  This was a DRY RUN - no data was written to silver table")
    print(f"Review results above before running STEP 6 to actually reprocess.")
    
    # Store for next step
    spark.catalog.dropTempView("fixed_quarantine_records")
    fixed_df.createOrReplaceTempView("fixed_quarantine_records")
    print(f"\n✅ Fixed records saved to temp view: fixed_quarantine_records")

In [0]:
# WRITE MODE: Actually insert fixed records into silver_telemetry_enriched
# Uses MERGE to prevent duplicates if records are reprocessed multiple times

print("⚠️  WARNING: This will MERGE data into silver_telemetry_enriched")
print("="*80)

# Safety check - require manual confirmation
CONFIRM_REPROCESS = False  # Set to True to enable reprocessing

if not CONFIRM_REPROCESS:
    print("\n🛑 Reprocessing is DISABLED")
    print("\nTo enable:")
    print("  1. Review the DRY RUN results in Step 5")
    print("  2. Set CONFIRM_REPROCESS = True in this cell")
    print("  3. Re-run this cell")
else:
    print("\n▶️  Reprocessing ENABLED - proceeding...\n")
    
    # Load fixed records from temp view
    try:
        fixed_df = spark.table("fixed_quarantine_records")
        record_count = fixed_df.count()
        
        print(f"📊 Records to reprocess: {record_count:,}\n")
        
        if record_count == 0:
            print("✅ No records to reprocess")
        else:
            # Create a temp view for the MERGE operation
            fixed_df.createOrReplaceTempView("quarantine_to_merge")
            
            print("Method: MERGE into silver_telemetry_enriched (prevents duplicates)\n")
            print("Merge key: customerId + clientId + timestamp_server + quality_level\n")
            
            # MERGE logic: Updates existing records, inserts new ones
            # This prevents duplicates if the same record is reprocessed multiple times
            merge_sql = f"""
            MERGE INTO {SILVER_TABLE} AS target
            USING quarantine_to_merge AS source
            ON target.customerId = source.customerId
               AND target.clientId = source.clientId
               AND target.timestamp_server = source.timestamp_server
               AND target.quality_level = source.quality_level
               AND target.eventDate = source.eventDate
            WHEN MATCHED THEN
              UPDATE SET *
            WHEN NOT MATCHED THEN
              INSERT *
            """
            
            # Execute MERGE
            spark.sql(merge_sql)
            
            print(f"✅ Successfully merged {record_count:,} records into silver")
            print("   - Existing records were UPDATED (no duplicates)")
            print("   - New records were INSERTED")
            
            # Show merge statistics
            print("\n📊 Merge Statistics:")
            print("   Use DESCRIBE HISTORY to see Delta merge metrics")
            
            # Mark quarantine records as processed
            print("\n📝 Next step: Mark these records as processed in quarantine table")
            print("   (See Step 8 for cleanup options)")
            
    except Exception as e:
        print(f"\n❌ Error during reprocessing: {str(e)}")
        print("\nTroubleshooting:")
        print("  1. Ensure Step 5 (DRY RUN) was executed successfully")
        print("  2. Check that temp view 'fixed_quarantine_records' exists")
        print("  3. Verify schema matches silver_telemetry_enriched")
        print("  4. Check that merge key columns exist in both tables")

## 🔄 Deduplication Strategy for Quarantine Reprocessing

### 🚨 The Problem: Why Deduplication Matters

**Scenario**: A quarantine record is reprocessed twice

```
Attempt 1: Record A fixed → INSERT into silver → Gold aggregates it
Attempt 2: Record A fixed again → INSERT into silver → Gold aggregates it AGAIN

Result in Gold:
  total_buffering_time = 2x actual value ❌
  qos_score = incorrectly calculated ❌
```

**Why this happens:**
1. **APPEND mode** creates duplicates in silver
2. **Materialized view** aggregates ALL records (including duplicates)
3. **SUM/COUNT** operations double-count the same session

### ✅ Solution 1: MERGE on Silver (IMPLEMENTED)

**Using MERGE instead of APPEND:**

```sql
MERGE INTO silver_telemetry_enriched AS target
USING fixed_quarantine_records AS source
ON target.customerId = source.customerId
   AND target.clientId = source.clientId
   AND target.timestamp_server = source.timestamp_server
   AND target.quality_level = source.quality_level
   AND target.eventDate = source.eventDate
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

**Merge Key** (natural key for deduplication):
- `customerId` + `clientId` + `timestamp_server` + `quality_level` + `eventDate`

**Behavior:**
- ✅ First reprocessing: Record inserted
- ✅ Second reprocessing: Record **updated** (not duplicated)
- ✅ Gold layer sees only ONE copy

### ✅ Solution 2: Add Deduplication to Gold (Optional)

If silver already has duplicates, add deduplication before aggregation:

```python
# In gold_viewer_qos_metrics.py
def gold_viewer_qos_metrics():
    df = spark.read.table("silver_telemetry_enriched")
    
    # Deduplicate before aggregation
    deduped = df.dropDuplicates([
        "customerId", "clientId", "timestamp_server", 
        "quality_level", "eventDate"
    ])
    
    # Then aggregate
    viewer_metrics = deduped.groupBy(...).agg(...)
```

**Trade-offs:**
- ✅ Handles historical duplicates
- ❌ Adds processing overhead
- ❌ Doesn't fix root cause (duplicates in silver)

### ✅ Solution 3: Track Processed Quarantine Records (Preventive)

Prevent double reprocessing at the source:

```python
# Add processed_id column to track what's been reprocessed
quarantine_df = quarantine_df.filter(
    F.col("processed_id").isNull()  # Only unprocessed records
)

# After successful reprocessing:
spark.sql(f"""
    UPDATE {QUARANTINE_TABLE}
    SET processed_id = uuid(), processed_timestamp = current_timestamp()
    WHERE customerId IN (...) AND eventDate IN (...)
""")
```

### 🎯 Recommended Strategy

**Use MERGE (Solution 1) + Tracking (Solution 3)**

1. **MERGE to silver** → Prevents duplicates if retry happens
2. **Track processed records** → Prevents unnecessary reprocessing
3. **Monitor silver for duplicates** → Alert if duplicates detected

### 🔍 How to Check for Existing Duplicates

Run this query to detect duplicates in silver:

```sql
SELECT 
  customerId,
  clientId,
  timestamp_server,
  quality_level,
  eventDate,
  COUNT(*) as duplicate_count
FROM workspace.hive_video_analytics.silver_telemetry_enriched
GROUP BY customerId, clientId, timestamp_server, quality_level, eventDate
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;
```

If duplicates exist:
1. They're already affecting gold metrics
2. Run one-time deduplication:
   ```sql
   CREATE OR REPLACE TABLE silver_telemetry_enriched AS
   SELECT DISTINCT * FROM silver_telemetry_enriched;
   ```
3. Gold materialized view will auto-refresh with correct data

### ⚠️ Important Notes

1. **MERGE is idempotent** → Safe to rerun
2. **Gold auto-refreshes** → No manual intervention needed
3. **Merge key must be unique** → Defines what's a "duplicate"
4. **Late data works correctly** → MERGE handles updates

## 🔄 How MERGE in Silver Propagates to Gold Layer

### 📊 The Data Flow

```
Quarantine reprocessing:
  1. MERGE INTO silver_telemetry_enriched  ← Updates/inserts records
  2. Delta log records the changes          ← Transaction tracking
  3. Gold materialized view detects changes ← Automatic detection
  4. Gold recomputes affected aggregates    ← Incremental refresh
```

### 🔍 Step-by-Step: What Happens During MERGE

**Example Scenario:**

You reprocess a quarantine record for `customer_123` on `2025-11-14`:

#### Step 1: MERGE Operation in Silver

```sql
MERGE INTO silver_telemetry_enriched AS target
USING quarantine_to_merge AS source
ON target.customerId = source.customerId
   AND target.clientId = source.clientId
   AND target.timestamp_server = source.timestamp_server
   AND target.quality_level = source.quality_level
   AND target.eventDate = source.eventDate
WHEN MATCHED THEN UPDATE SET *      ← Updates existing record
WHEN NOT MATCHED THEN INSERT *      ← Or inserts new record
```

**What Delta Lake records:**
- Transaction timestamp
- Files modified (data files for eventDate=2025-11-14)
- Operation type (MERGE)
- Number of rows updated vs inserted

#### Step 2: Delta Transaction Log

```
Delta Log Entry:
{
  "commitInfo": {
    "timestamp": 1732464000000,
    "operation": "MERGE",
    "operationMetrics": {
      "numTargetRowsUpdated": 5,    ← These 5 records changed
      "numTargetRowsInserted": 0,
      "numTargetFilesAdded": 1,
      "numTargetPartitionsUpdated": 1  ← eventDate=2025-11-14 partition
    }
  }
}
```

#### Step 3: Materialized View Change Detection

**Materialized view monitors silver's Delta log:**

```python
# Internally, the MV framework does something like:
last_processed_version = get_last_processed_version()
current_version = silver_table.history().first().version

if current_version > last_processed_version:
    changed_partitions = detect_changed_partitions(silver_table)
    # Returns: [(eventDate=2025-11-14, files_changed)]
```

**How it detects changes:**
- ✅ Reads Delta log incrementally (not full table scan)
- ✅ Identifies which `eventDate` partitions were modified
- ✅ Works for INSERT, UPDATE (from MERGE), DELETE, or any operation

#### Step 4: Gold Layer Incremental Refresh

**Gold recomputes ONLY affected aggregates:**

```python
# Gold materialized view automatically does:
changed_dates = [2025-11-14]  # From Delta log

# Read ONLY changed partitions from silver
silver_changed = spark.read.table("silver_telemetry_enriched") \
    .filter(F.col("eventDate").isin(changed_dates))

# Recompute aggregates for those dates
new_aggregates = silver_changed.groupBy(
    "customerId", "clientId", "eventDate"
).agg(
    F.sum("buffering_count").alias("total_buffering_events"),
    F.sum("buffering_time_sec").alias("total_buffering_time_sec"),
    ...
)

# MERGE these back into gold (replace old aggregates for 2025-11-14)
spark.sql("""
    MERGE INTO gold_viewer_qos_metrics AS target
    USING new_aggregates AS source
    ON target.customerId = source.customerId
       AND target.clientId = source.clientId
       AND target.eventDate = source.eventDate
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
```

**Result:**
- ✅ Gold aggregates for `2025-11-14` are **recalculated** with updated silver data
- ✅ Aggregates for other dates remain unchanged (no wasted computation)
- ✅ Gold always reflects the **current state** of silver

### 🎯 Key Points

**1. MERGE is Just Another Delta Operation**
- Delta Lake treats MERGE like any other change
- Creates transaction log entries
- Modifies data files for affected partitions

**2. Materialized View Detects ALL Changes**
- Monitors Delta log continuously
- Detects INSERT, UPDATE, DELETE, MERGE
- No special handling needed for MERGE vs INSERT

**3. Incremental Refresh is Automatic**
- You don't trigger refresh manually
- Happens automatically when pipeline runs
- Or can be scheduled separately

**4. Liquid Clustering Enables Efficiency**
- `[eventDate, customerId, clientId]` clustering
- Allows pinpoint refresh of affected partitions
- Only changed dates are reprocessed

### 📈 Performance: MERGE vs INSERT Impact

**Scenario: Reprocess 1,000 quarantine records for one date**

| Operation | Silver Cost | Gold Refresh Cost |
|-----------|-------------|-------------------|
| **1st time** (INSERT) | Write 1,000 new rows | Aggregate 1,000 rows for that date |
| **2nd time** (MERGE UPDATE) | Update 1,000 existing rows | **Same** - recompute aggregates for that date |
| **Difference** | Slightly slower (read+write vs write) | **No difference** - same aggregation cost |

**Gold refresh cost is the SAME** whether silver did INSERT or MERGE:
- Both modify the same `eventDate` partition
- Gold recomputes that partition either way
- MERGE vs INSERT doesn't matter to gold

### ⚠️ Important: MERGE Doesn't "Propagate" Operations

**Common Misconception:**
```
❌ WRONG: "Gold needs to run MERGE because silver ran MERGE"
✅ RIGHT: "Gold recomputes aggregates because silver data changed"
```

**Gold doesn't replicate MERGE operations:**
- Gold doesn't care HOW silver changed (MERGE, INSERT, UPDATE, DELETE)
- Gold only cares WHICH partitions changed
- Gold always does the same thing: aggregate the current silver data

### 🔍 How to Verify MERGE Propagation

**1. Check Delta History**
```sql
-- See MERGE operation in silver
DESCRIBE HISTORY workspace.hive_video_analytics.silver_telemetry_enriched
ORDER BY version DESC LIMIT 5;

-- See refresh operation in gold
DESCRIBE HISTORY workspace.hive_video_analytics.gold_viewer_qos_metrics
ORDER BY version DESC LIMIT 5;
```

**2. Verify Data Consistency**
```sql
-- Compare silver vs gold aggregates for a date
WITH silver_agg AS (
  SELECT 
    customerId, clientId, eventDate,
    SUM(buffering_count) as silver_total_buffering
  FROM workspace.hive_video_analytics.silver_telemetry_enriched
  WHERE eventDate = '2025-11-14'
  GROUP BY customerId, clientId, eventDate
)
SELECT 
  s.customerId,
  s.silver_total_buffering,
  g.total_buffering_events as gold_total_buffering,
  CASE 
    WHEN s.silver_total_buffering = g.total_buffering_events THEN '✅ Match'
    ELSE '❌ Mismatch'
  END as status
FROM silver_agg s
JOIN workspace.hive_video_analytics.gold_viewer_qos_metrics g
  ON s.customerId = g.customerId 
  AND s.clientId = g.clientId
  AND s.eventDate = g.eventDate;
```

### 🚀 When Does Gold Refresh Happen?

**Automatic triggers:**
1. **Pipeline runs** - Gold refreshes at end of pipeline execution
2. **Scheduled refresh** - Can be configured separately
3. **Manual refresh** - Run pipeline update explicitly

**To manually trigger gold refresh:**
```python
# Option 1: Via pipeline update
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
w.pipelines.start_update(
    pipeline_id="4296766b-887a-4ab4-9eef-2e6e07a32b66",
    full_refresh=False  # Incremental refresh
)

# Option 2: Via REFRESH MATERIALIZED VIEW (if standalone)
spark.sql("REFRESH MATERIALIZED VIEW workspace.hive_video_analytics.gold_viewer_qos_metrics")
```

### ✅ Summary

**MERGE in silver automatically flows to gold because:**

1. **Delta Lake** records all changes in transaction log
2. **Materialized View** monitors Delta log for changes
3. **Incremental refresh** recomputes affected partitions
4. **No manual intervention** needed

**The beauty of this architecture:**
- ✅ Write MERGE once (in silver)
- ✅ Gold updates automatically
- ✅ No duplicate logic needed
- ✅ Efficient (only changed partitions refresh)
- ✅ Correct (always reflects current silver state)

In [0]:
%sql
-- Verify that MERGE operations in silver are reflected in gold
-- This query shows the refresh synchronization between layers

WITH silver_history AS (
  SELECT 
    'SILVER' as layer,
    version,
    timestamp,
    operation,
    operationMetrics.numTargetRowsUpdated as rows_updated,
    operationMetrics.numTargetRowsInserted as rows_inserted,
    operationMetrics.numOutputRows as total_rows
  FROM (DESCRIBE HISTORY workspace.hive_video_analytics.silver_telemetry_enriched)
  WHERE operation IN ('MERGE', 'WRITE', 'UPDATE')
  ORDER BY version DESC
  LIMIT 5
),
gold_history AS (
  SELECT 
    'GOLD' as layer,
    version,
    timestamp,
    operation,
    operationMetrics.numTargetRowsUpdated as rows_updated,
    operationMetrics.numTargetRowsInserted as rows_inserted,
    operationMetrics.numOutputRows as total_rows
  FROM (DESCRIBE HISTORY workspace.hive_video_analytics.gold_viewer_qos_metrics)
  WHERE operation IN ('MERGE', 'WRITE', 'UPDATE')
  ORDER BY version DESC
  LIMIT 5
)
SELECT 
  layer,
  version,
  timestamp,
  operation,
  COALESCE(rows_updated, 0) as rows_updated,
  COALESCE(rows_inserted, 0) as rows_inserted,
  COALESCE(total_rows, 0) as total_rows
FROM silver_history

UNION ALL

SELECT 
  layer,
  version,
  timestamp,
  operation,
  COALESCE(rows_updated, 0) as rows_updated,
  COALESCE(rows_inserted, 0) as rows_inserted,
  COALESCE(total_rows, 0) as total_rows
FROM gold_history

ORDER BY timestamp DESC;

-- Expected output:
-- You'll see MERGE operations in silver followed by
-- corresponding WRITE/MERGE operations in gold
-- (gold refresh happens after silver changes)

In [0]:
%sql
-- Check if silver layer already has duplicates from previous reprocessing
-- These would cause incorrect gold aggregations

WITH duplicate_check AS (
  SELECT 
    customerId,
    clientId,
    timestamp_server,
    quality_level,
    eventDate,
    COUNT(*) as duplicate_count
  FROM workspace.hive_video_analytics.silver_telemetry_enriched
  GROUP BY customerId, clientId, timestamp_server, quality_level, eventDate
  HAVING COUNT(*) > 1
)
SELECT 
  'Duplicate Detection Summary' as check_type,
  COUNT(*) as duplicate_groups,
  SUM(duplicate_count) as total_duplicate_records,
  SUM(duplicate_count - 1) as extra_copies
FROM duplicate_check

UNION ALL

SELECT 
  'Sample Duplicates (Top 5)' as check_type,
  NULL,
  NULL,
  NULL
FROM (SELECT 1) -- Dummy to maintain schema

UNION ALL

SELECT 
  CONCAT('  ', customerId, ' | ', clientId, ' | ', DATE(FROM_UNIXTIME(timestamp_server/1000))) as check_type,
  NULL as duplicate_groups,
  duplicate_count as total_duplicate_records,
  NULL as extra_copies
FROM duplicate_check
ORDER BY total_duplicate_records DESC NULLS LAST
LIMIT 10;

In [0]:
# One-time cleanup: Remove duplicates from silver layer
# Only run if duplicates were detected above

print("🧹 Silver Layer Deduplication\n")
print("="*80)

FIX_DUPLICATES = False  # Set to True to enable cleanup

if not FIX_DUPLICATES:
    print("\n🛑 Deduplication is DISABLED")
    print("\nTo enable:")
    print("  1. Run the 'Detect Existing Duplicates' cell above")
    print("  2. Review the duplicate count")
    print("  3. Set FIX_DUPLICATES = True in this cell")
    print("  4. Re-run this cell")
    print("\n⚠️  WARNING: This will rewrite the entire silver table")
else:
    print("\n▶️  Deduplication ENABLED - proceeding...\n")
    
    # Read silver table
    silver_df = spark.table(SILVER_TABLE)
    original_count = silver_df.count()
    print(f"📊 Original silver records: {original_count:,}\n")
    
    # Define unique key for deduplication
    unique_cols = [
        "customerId", "clientId", "timestamp_server", 
        "quality_level", "eventDate"
    ]
    
    print(f"🔑 Deduplication key: {', '.join(unique_cols)}\n")
    
    # Method 1: Keep first occurrence (faster)
    print("Method: DROP_DUPLICATES (keeps first occurrence)\n")
    deduped_df = silver_df.dropDuplicates(unique_cols)
    
    deduped_count = deduped_df.count()
    duplicates_removed = original_count - deduped_count
    
    print(f"📊 Deduplication results:")
    print(f"   Original records:     {original_count:,}")
    print(f"   Deduplicated records: {deduped_count:,}")
    print(f"   Duplicates removed:   {duplicates_removed:,}")
    
    if duplicates_removed == 0:
        print("\n✅ No duplicates found - silver layer is clean!")
    else:
        print(f"\n⚠️  Found {duplicates_removed:,} duplicate records\n")
        
        # Create backup before overwriting
        BACKUP_TABLE = f"{SILVER_TABLE}_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        print(f"📦 Creating backup: {BACKUP_TABLE}")
        silver_df.write.format("delta").saveAsTable(BACKUP_TABLE)
        print("✅ Backup created\n")
        
        # Overwrite silver with deduplicated data
        print(f"🔄 Overwriting silver table with deduplicated data...")
        deduped_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "false") \
            .saveAsTable(SILVER_TABLE)
        
        print("✅ Silver table deduplicated successfully!")
        print(f"\n📊 Final silver record count: {deduped_count:,}")
        print(f"\n🔄 Gold materialized view will auto-refresh with correct data")
        print(f"   Run the pipeline to trigger gold layer refresh")
        
        print(f"\n📦 Backup location: {BACKUP_TABLE}")
        print(f"   Keep for 30 days, then drop if no issues")
        print(f"   DROP TABLE {BACKUP_TABLE};")

In [0]:
%sql
-- Verify that reprocessed records are now in silver layer
-- Compare record counts before and after

WITH silver_counts AS (
  SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT customerId) as unique_customers,
    MIN(eventDate) as earliest_date,
    MAX(eventDate) as latest_date
  FROM workspace.hive_video_analytics.silver_telemetry_enriched
),
quarantine_remaining AS (
  SELECT 
    COUNT(*) as remaining_quarantine,
    COUNT(DISTINCT customerId) as affected_customers
  FROM workspace.hive_video_analytics.silver_telemetry_quarantine
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
)
SELECT 
  'Silver Enriched' as layer,
  total_records as records,
  unique_customers as customers,
  CONCAT(earliest_date, ' to ', latest_date) as date_range
FROM silver_counts

UNION ALL

SELECT
  'Quarantine (last 7d)' as layer,
  remaining_quarantine as records,
  affected_customers as customers,
  'Recent records only' as date_range
FROM quarantine_remaining;

In [0]:
# Clean up quarantine table after successful reprocessing
# Options: DELETE processed records OR mark as processed

print("🗄️ Quarantine Record Cleanup Options\n")
print("="*80)

CLEANUP_MODE = "none"  # Options: "none", "mark_processed", "delete", "archive"

print(f"Current mode: {CLEANUP_MODE}\n")

if CLEANUP_MODE == "none":
    print("No cleanup configured. Choose an option:\n")
    print("1. 'mark_processed' - Add processed_timestamp column")
    print("   Pros: Keeps audit trail, can review later")
    print("   Cons: Quarantine table grows over time\n")
    
    print("2. 'delete' - DELETE reprocessed records")
    print("   Pros: Keeps quarantine small, only current issues")
    print("   Cons: Loses history of what was fixed\n")
    
    print("3. 'archive' - Move to quarantine_archive table")
    print("   Pros: Best of both - clean quarantine, keep history")
    print("   Cons: Need to create archive table\n")
    
    print("Set CLEANUP_MODE to your preferred option and re-run.")

elif CLEANUP_MODE == "mark_processed":
    print("🏷️ Marking records as processed...\n")
    
    # Add processed_timestamp column if not exists
    quarantine_df = spark.table(QUARANTINE_TABLE)
    if "processed_timestamp" not in quarantine_df.columns:
        spark.sql(f"""
            ALTER TABLE {QUARANTINE_TABLE} 
            ADD COLUMN processed_timestamp TIMESTAMP
        """)
        print("✅ Added processed_timestamp column")
    
    # Update records that were reprocessed
    # NOTE: This requires tracking which records were processed
    print("⚠️ Implementation needed: Update processed_timestamp for reprocessed records")
    print("   Use primary key (customerId, eventDate, timestamp) to match records")

elif CLEANUP_MODE == "delete":
    print("🗑️ Deleting reprocessed records...\n")
    
    DELETE_CONFIRM = False  # Set to True to enable deletion
    
    if not DELETE_CONFIRM:
        print("🛑 Deletion is DISABLED (DELETE_CONFIRM = False)")
        print("   Set DELETE_CONFIRM = True to enable")
    else:
        # Example: Delete records older than 7 days that match reprocessed records
        # NOTE: Adjust WHERE clause based on how you track processed records
        print("⚠️ Implementation needed: DELETE reprocessed quarantine records")
        print("   Use safe WHERE clause with date and customer filters")

elif CLEANUP_MODE == "archive":
    print("📦 Archiving to quarantine_archive table...\n")
    
    ARCHIVE_TABLE = f"{CATALOG}.{SCHEMA}.silver_telemetry_quarantine_archive"
    
    # Create archive table if not exists
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {ARCHIVE_TABLE}
        LIKE {QUARANTINE_TABLE}
    """)
    print(f"✅ Archive table ready: {ARCHIVE_TABLE}")
    
    # Move processed records to archive
    print("⚠️ Implementation needed: INSERT INTO archive and DELETE FROM quarantine")
    print("   Use transaction to ensure atomic move")

print("\n📝 Recommendation: Use 'archive' mode for production")
print("   Keeps quarantine table small while maintaining full audit trail")

## 🤖 Automated Quarantine Reprocessing Workflow

### Schedule This Notebook

For continuous quarantine monitoring and reprocessing:

1. **Schedule frequency**: Daily or weekly
2. **Cluster**: Use same cluster as main pipeline
3. **Notifications**: Alert on failure
4. **Parameters**: 
   - `lookback_days`: How far back to process
   - `dry_run`: Set to False for production
   - `cleanup_mode`: Choose cleanup strategy

### Integration with Pipeline

**Option A: Separate Quarantine Job**
```
Main Pipeline Job:
  - Bronze → Silver → Gold
  - Quarantine bad records
  
Quarantine Processing Job (scheduled separately):
  - Analyze quarantine
  - Fix and reprocess
  - Clean up quarantine table
```

**Option B: Add Reprocessing Task to Pipeline**
```yaml
tasks:
  - task_key: main_pipeline
    pipeline_task:
      pipeline_id: xxx
  
  - task_key: quarantine_reprocessing
    depends_on:
      - task_key: main_pipeline
    notebook_task:
      notebook_path: operations/Quarantine_Data_Processing
    depends_on:
      - task_key: main_pipeline
```

### Monitoring

Create alerts for:
* Quarantine rate exceeds threshold (5%)
* Quarantine records older than 7 days
* Reprocessing success rate drops below 80%
* Permanent failures accumulating

### Best Practices

1. ✅ **Always run DRY RUN first** before reprocessing
2. ✅ **Archive processed records** for audit trail
3. ✅ **Monitor quarantine patterns** to fix root causes
4. ✅ **Set success rate targets** (e.g., 90% reprocessing success)
5. ✅ **Review permanent failures** quarterly to update fix logic
6. ✅ **Alert on new quarantine reasons** for investigation

In [0]:
%sql
-- Track quarantine rates over time to identify data quality trends
-- Use this to measure impact of fixes and identify new issues early

WITH daily_stats AS (
  SELECT 
    eventDate,
    quarantine_reason,
    COUNT(*) as quarantine_count
  FROM workspace.hive_video_analytics.silver_telemetry_quarantine
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 30 DAYS
  GROUP BY eventDate, quarantine_reason
),
bronze_daily AS (
  SELECT 
    eventDate,
    COUNT(*) as bronze_count
  FROM workspace.hive_video_analytics.bronze_telemetry_raw
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 30 DAYS
  GROUP BY eventDate
)
SELECT 
  daily_stats.eventDate,
  daily_stats.quarantine_reason,
  daily_stats.quarantine_count,
  bronze_daily.bronze_count,
  ROUND(daily_stats.quarantine_count * 100.0 / NULLIF(bronze_daily.bronze_count, 0), 2) as quarantine_rate_pct
FROM daily_stats
LEFT JOIN bronze_daily ON daily_stats.eventDate = bronze_daily.eventDate
ORDER BY daily_stats.eventDate DESC, daily_stats.quarantine_count DESC;